<a href="https://colab.research.google.com/github/WellingtonRoque/MineracaoDados/blob/main/atividades/aula7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

3. Criando um banco SQLite
Para não depender de instalação ou servidor externo, utilizaremos o SQLite.

Ele já está disponível no Python.

Em projetos profissionais, os mesmos conceitos podem ser aplicados a MySQL, PostgreSQL, SQL Server e outros bancos relacionais.

In [1]:
import sqlite3
import pandas as pd

conexao = sqlite3.connect("industria4.db")

# Ativa a verificação das chaves estrangeiras no SQLite
conexao.execute("PRAGMA foreign_keys = ON")

print("Banco de dados conectado!")

Banco de dados conectado!


5. Criando as tabelas
Vamos criar três tabelas:

motores
leituras
manutencoes
Elas representam diferentes fontes de informação da nossa indústria.

In [2]:
conexao.execute("""
CREATE TABLE IF NOT EXISTS motores (
    id INTEGER PRIMARY KEY,
    motor TEXT NOT NULL,
    modelo TEXT,
    fabricante TEXT,
    potencia_nominal REAL,
    linha TEXT
)
""")

conexao.commit()

print("Tabela motores pronta.")

Tabela motores pronta.


In [3]:
conexao.execute("""
CREATE TABLE IF NOT EXISTS leituras (
    id INTEGER PRIMARY KEY,
    motor_id INTEGER,
    data_hora TEXT,
    corrente REAL,
    tensao REAL,
    vibracao REAL,
    temperatura REAL,
    rpm INTEGER,
    FOREIGN KEY (motor_id) REFERENCES motores(id)
)
""")

conexao.commit()

print("Tabela leituras pronta.")

Tabela leituras pronta.


In [4]:

conexao.execute("""
CREATE TABLE IF NOT EXISTS manutencoes (
    id INTEGER PRIMARY KEY,
    motor_id INTEGER,
    data_manutencao TEXT,
    horas_operacao INTEGER,
    status TEXT,
    FOREIGN KEY (motor_id) REFERENCES motores(id)
)
""")

conexao.commit()

print("Tabela manutencoes pronta.")

Tabela manutencoes pronta.


Verificando as tabelas
Vamos conferir se as três tabelas existem no banco.

In [5]:
pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""", conexao)

,name
0,leituras
1,manutencoes
2,motores


6. Inserindo os dados
Os dados representam quatro motores da nossa indústria.

O INSERT OR IGNORE foi utilizado para que a célula possa ser executada novamente sem gerar erro de chave primária duplicada.

In [6]:
motores = [
    (1, "M001", "MX-100", "MotorTech", 7.5, "Linha A"),
    (2, "M002", "MX-100", "MotorTech", 6.5, "Linha A"),
    (3, "M003", "MX-200", "PowerMotor", 9.0, "Linha B"),
    (4, "M004", "MX-300", "PowerMotor", 11.0, "Linha B")
]

conexao.executemany(
    """
    INSERT OR IGNORE INTO motores
    (id, motor, modelo, fabricante, potencia_nominal, linha)
    VALUES (?, ?, ?, ?, ?, ?)
    """,
    motores
)

conexao.commit()

print("Dados de motores inseridos/verificados.")

Dados de motores inseridos/verificados.


In [7]:
leituras = [
    (1, 1, "2026-03-01 08:00", 12.4, 380, 1.8, 62.3, 1750),
    (2, 1, "2026-03-01 09:00", 12.8, 379, 2.1, 64.1, 1748),
    (3, 1, "2026-03-01 10:00", 13.1, 380, 2.5, 66.2, 1745),
    (4, 2, "2026-03-01 08:00", 10.8, 380, 1.2, 55.1, 1752),
    (5, 2, "2026-03-01 09:00", 11.0, 381, 1.3, 56.0, 1750),
    (6, 2, "2026-03-01 10:00", 11.2, 380, 1.5, 57.4, 1748),
    (7, 3, "2026-03-01 08:00", 14.1, 382, 2.4, 67.2, 1740),
    (8, 3, "2026-03-01 09:00", 14.3, 381, 2.6, 68.4, 1738),
    (9, 3, "2026-03-01 10:00", 14.8, 380, 2.9, 70.1, 1735),
    (10, 4, "2026-03-01 08:00", 16.2, 381, 3.1, 72.4, 1728),
    (11, 4, "2026-03-01 09:00", 16.8, 380, 3.5, 75.2, 1724),
    (12, 4, "2026-03-01 10:00", 17.4, 379, 3.8, 78.1, 1720)
]

conexao.executemany(
    """
    INSERT OR IGNORE INTO leituras
    (id, motor_id, data_hora, corrente, tensao, vibracao, temperatura, rpm)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """,
    leituras
)

conexao.commit()

print("Dados de leituras inseridos/verificados.")

Dados de leituras inseridos/verificados.


In [8]:
manutencoes = [
    (1, 1, "2026-02-15", 1254, "Em dia"),
    (2, 2, "2026-02-20", 824, "Em dia"),
    (3, 3, "2026-01-18", 2104, "Atenção"),
    (4, 4, "2026-01-10", 2980, "Atenção")
]

conexao.executemany(
    """
    INSERT OR IGNORE INTO manutencoes
    (id, motor_id, data_manutencao, horas_operacao, status)
    VALUES (?, ?, ?, ?, ?)
    """,
    manutencoes
)

conexao.commit()

print("Dados de manutenção inseridos/verificados.")

Dados de manutenção inseridos/verificados.


Conferindo a quantidade de registros
Esta etapa ajuda a verificar se os dados foram carregados corretamente.

In [9]:
resumo = pd.read_sql_query("""
SELECT
    (SELECT COUNT(*) FROM motores) AS total_motores,
    (SELECT COUNT(*) FROM leituras) AS total_leituras,
    (SELECT COUNT(*) FROM manutencoes) AS total_manutencoes
""", conexao)

resumo

,total_motores,total_leituras,total_manutencoes
0,4,12,4


7. SELECT — Consultando dados
O comando mais básico do SQL é SELECT.

In [10]:
consulta = """
SELECT *
FROM motores
"""

pd.read_sql_query(consulta, conexao)

,id,motor,modelo,fabricante,potencia_nominal,linha
0,1,M001,MX-100,MotorTech,7.5,Linha A
1,2,M002,MX-100,MotorTech,6.5,Linha A
2,3,M003,MX-200,PowerMotor,9.0,Linha B
3,4,M004,MX-300,PowerMotor,11.0,Linha B


In [11]:
consulta = """
SELECT motor, modelo, potencia_nominal
FROM motores
"""

pd.read_sql_query(consulta, conexao)

,motor,modelo,potencia_nominal
0,M001,MX-100,7.5
1,M002,MX-100,6.5
2,M003,MX-200,9.0
3,M004,MX-300,11.0


 8. WHERE — Filtrando dados
O WHERE permite selecionar registros que atendam a uma condição.

In [14]:
consulta = """
SELECT *
FROM motores
WHERE linha = 'Linha A'
"""

pd.read_sql_query(consulta, conexao)

,id,motor,modelo,fabricante,potencia_nominal,linha
0,1,M001,MX-100,MotorTech,7.5,Linha A
1,2,M002,MX-100,MotorTech,6.5,Linha A


In [15]:
consulta = """
SELECT motor, potencia_nominal
FROM motores
WHERE potencia_nominal > 7
"""

pd.read_sql_query(consulta, conexao)

,motor,potencia_nominal
0,M001,7.5
1,M003,9.0
2,M004,11.0


9. ORDER BY — Ordenando resultados

In [16]:
consulta = """
SELECT motor, potencia_nominal
FROM motores
ORDER BY potencia_nominal DESC
"""

pd.read_sql_query(consulta, conexao)

,motor,potencia_nominal
0,M004,11.0
1,M003,9.0
2,M001,7.5
3,M002,6.5


10. Funções de agregação
SQL possui funções para realizar cálculos:

In [17]:
consulta = """
SELECT AVG(temperatura) AS temperatura_media
FROM leituras
"""

pd.read_sql_query(consulta, conexao)

,temperatura_media
0,66.041667


In [18]:
consulta = """
SELECT
    MIN(temperatura) AS temperatura_minima,
    MAX(temperatura) AS temperatura_maxima,
    AVG(temperatura) AS temperatura_media,
    AVG(vibracao) AS vibracao_media
FROM leituras
"""

pd.read_sql_query(consulta, conexao)

,temperatura_minima,temperatura_maxima,temperatura_media,vibracao_media
0,55.1,78.1,66.041667,2.391667


11. GROUP BY
Agora queremos saber a temperatura média de cada motor.

In [19]:
consulta = """
SELECT
    motor_id,
    AVG(temperatura) AS temperatura_media
FROM leituras
GROUP BY motor_id
ORDER BY temperatura_media DESC
"""

pd.read_sql_query(consulta, conexao)

,motor_id,temperatura_media
0,4,75.233333
1,3,68.566667
2,1,64.200000
3,2,56.166667


12. JOIN — Relacionando tabelas
Até agora leituras possui apenas motor_id.

Queremos descobrir o nome do motor. Para isso, precisamos relacionar as tabelas.

In [20]:
consulta = """
SELECT
    m.motor,
    m.modelo,
    l.data_hora,
    l.temperatura,
    l.vibracao,
    l.corrente
FROM leituras AS l
INNER JOIN motores AS m
    ON l.motor_id = m.id
"""

pd.read_sql_query(consulta, conexao)

,motor,modelo,data_hora,temperatura,vibracao,corrente
0,M001,MX-100,2026-03-01 08:00,62.3,1.8,12.4
1,M001,MX-100,2026-03-01 09:00,64.1,2.1,12.8
2,M001,MX-100,2026-03-01 10:00,66.2,2.5,13.1
3,M002,MX-100,2026-03-01 08:00,55.1,1.2,10.8
4,M002,MX-100,2026-03-01 09:00,56.0,1.3,11.0
5,M002,MX-100,2026-03-01 10:00,57.4,1.5,11.2
6,M003,MX-200,2026-03-01 08:00,67.2,2.4,14.1
7,M003,MX-200,2026-03-01 09:00,68.4,2.6,14.3
8,M003,MX-200,2026-03-01 10:00,70.1,2.9,14.8
9,M004,MX-300,2026-03-01 08:00,72.4,3.1,16.2


 13. Criando uma base para Mineração de Dados
Agora vamos produzir uma tabela integrada com informações das três tabelas.

In [21]:
consulta = """
SELECT
    m.motor,
    m.modelo,
    m.fabricante,
    m.linha,
    l.data_hora,
    l.corrente,
    l.tensao,
    l.vibracao,
    l.temperatura,
    l.rpm,
    man.horas_operacao,
    man.status AS status_manutencao
FROM leituras AS l
INNER JOIN motores AS m
    ON l.motor_id = m.id
LEFT JOIN manutencoes AS man
    ON m.id = man.motor_id
ORDER BY m.motor, l.data_hora
"""

dados_mineracao = pd.read_sql_query(consulta, conexao)

dados_mineracao

,motor,modelo,fabricante,linha,data_hora,corrente,tensao,vibracao,temperatura,rpm,horas_operacao,status_manutencao
0,M001,MX-100,MotorTech,Linha A,2026-03-01 08:00,12.4,380.0,1.8,62.3,1750,1254,Em dia
1,M001,MX-100,MotorTech,Linha A,2026-03-01 09:00,12.8,379.0,2.1,64.1,1748,1254,Em dia
2,M001,MX-100,MotorTech,Linha A,2026-03-01 10:00,13.1,380.0,2.5,66.2,1745,1254,Em dia
3,M002,MX-100,MotorTech,Linha A,2026-03-01 08:00,10.8,380.0,1.2,55.1,1752,824,Em dia
4,M002,MX-100,MotorTech,Linha A,2026-03-01 09:00,11.0,381.0,1.3,56.0,1750,824,Em dia
5,M002,MX-100,MotorTech,Linha A,2026-03-01 10:00,11.2,380.0,1.5,57.4,1748,824,Em dia
6,M003,MX-200,PowerMotor,Linha B,2026-03-01 08:00,14.1,382.0,2.4,67.2,1740,2104,Atenção
7,M003,MX-200,PowerMotor,Linha B,2026-03-01 09:00,14.3,381.0,2.6,68.4,1738,2104,Atenção
8,M003,MX-200,PowerMotor,Linha B,2026-03-01 10:00,14.8,380.0,2.9,70.1,1735,2104,Atenção
9,M004,MX-300,PowerMotor,Linha B,2026-03-01 08:00,16.2,381.0,3.1,72.4,1728,2980,Atenção


14. SQL + Pandas
SQL é excelente para:

selecionar;

filtrar;

relacionar;

agrupar.


Pandas é excelente para:

limpar;

transformar;

explorar;

analisar;

visualizar.

As duas ferramentas podem trabalhar juntas.

In [22]:
dados_mineracao.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   motor              12 non-null     object 
 1   modelo             12 non-null     object 
 2   fabricante         12 non-null     object 
 3   linha              12 non-null     object 
 4   data_hora          12 non-null     object 
 5   corrente           12 non-null     float64
 6   tensao             12 non-null     float64
 7   vibracao           12 non-null     float64
 8   temperatura        12 non-null     float64
 9   rpm                12 non-null     int64  
 10  horas_operacao     12 non-null     int64  
 11  status_manutencao  12 non-null     object 
dtypes: float64(4), int64(2), object(6)
memory usage: 1.3+ KB


In [23]:
dados_mineracao.describe()

,corrente,tensao,vibracao,temperatura,rpm,horas_operacao
count,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000
mean,13.741667,380.250000,2.391667,66.041667,1739.833333,1790.500000
std,2.257697,0.866025,0.845801,7.422994,10.977939,863.683496
min,10.800000,379.000000,1.200000,55.100000,1720.000000,824.000000
25%,12.100000,380.000000,1.725000,61.075000,1733.250000,1146.500000
50%,13.600000,380.000000,2.450000,66.700000,1742.500000,1679.000000
75%,15.150000,381.000000,2.950000,70.675000,1748.500000,2323.000000
max,17.400000,382.000000,3.800000,78.100000,1752.000000,2980.000000


15. Investigando o problema
Vamos procurar motores que apresentem temperatura acima de 70 °C.

In [24]:
consulta = """
SELECT
    m.motor,
    l.data_hora,
    l.temperatura,
    l.vibracao
FROM leituras AS l
INNER JOIN motores AS m
    ON l.motor_id = m.id
WHERE l.temperatura > 70
ORDER BY l.temperatura DESC
"""

pd.read_sql_query(consulta, conexao)

,motor,data_hora,temperatura,vibracao
0,M004,2026-03-01 10:00,78.1,3.8
1,M004,2026-03-01 09:00,75.2,3.5
2,M004,2026-03-01 08:00,72.4,3.1
3,M003,2026-03-01 10:00,70.1,2.9


Agora podemos combinar duas condições:

temperatura > 70
E
vibracao > 3

In [25]:
consulta = """
SELECT
    m.motor,
    l.data_hora,
    l.temperatura,
    l.vibracao
FROM leituras AS l
INNER JOIN motores AS m
    ON l.motor_id = m.id
WHERE l.temperatura > 70
  AND l.vibracao > 3
ORDER BY l.temperatura DESC
"""

pd.read_sql_query(consulta, conexao)

,motor,data_hora,temperatura,vibracao
0,M004,2026-03-01 10:00,78.1,3.8
1,M004,2026-03-01 09:00,75.2,3.5
2,M004,2026-03-01 08:00,72.4,3.1
